# text2sql QLoRA fine-tune — free Colab T4

**Phone steps:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Add your tokens in the **Secrets** panel (🔑 left sidebar):
- `HF_TOKEN` — required (gated Llama-3.1 access)
- `GH_TOKEN` — only if this repo is **private**

Run the **smoke** cell first (~minutes) to confirm it works, then the **full** cell.


In [ ]:
# 1. Confirm the T4
!nvidia-smi -L
import torch; print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))


In [ ]:
# 2. Clone the repo (public clone; falls back to GH_TOKEN secret if private)
import os, subprocess
REPO   = "https://github.com/mohamed-ahmed-58059/hf-ml-platform-text2sql.finetune"
BRANCH = "claude/colab-cli-t4-gpu-8z1v8i"
DIR    = "/content/text2sql.finetune"
gh = None
try:
    from google.colab import userdata
    gh = userdata.get('GH_TOKEN')
except Exception:
    pass
url = REPO.replace("https://", f"https://{gh}@") if gh else REPO
if not os.path.exists(DIR):
    subprocess.run(["git","clone","--branch",BRANCH,"--depth","1",url,DIR], check=True)
%cd /content/text2sql.finetune


In [ ]:
# 3. Install training deps (torch already on the T4)
!pip install -q -r colab/requirements-train.txt


In [ ]:
# 4. HuggingFace login (gated Llama-3.1)
import os
from google.colab import userdata
from huggingface_hub import login
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(os.environ["HF_TOKEN"])


## Smoke run — validate end-to-end (~minutes)


In [ ]:
# 5. SMOKE: tiny subset + few steps
import os
os.environ.update({"T2S_SMOKE":"true","T2S_RESUME_FROM":"",
                   "T2S_LOG_TO_WANDB":"false","T2S_PUSH_TO_HUB":"false"})
!python train.py


## Full run
Free T4 is slow and may disconnect; this pushes a checkpoint to the HF Hub every save,
so a disconnect loses nothing — resume by setting `T2S_RESUME_FROM=out/<run>/checkpoint-N`.


In [ ]:
# 6. FULL: real fine-tune, checkpoints streamed to the HF Hub
import os
os.environ.update({"T2S_SMOKE":"false","T2S_RESUME_FROM":"",
                   "T2S_LOG_TO_WANDB":"false","T2S_PUSH_TO_HUB":"true"})
!python train.py
